In [1]:
import pandas as pd

df = pd.read_csv("SMSSpamCollection_cleaned.csv")

print(df.head())


print("Dataset shape:", df.shape)


print(df["label"].value_counts())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
Dataset shape: (5169, 2)
label
ham     4516
spam     653
Name: count, dtype: int64


In [5]:
from sklearn.model_selection import train_test_split

X = df["message"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
X,
y,
test_size=0.2,
random_state=42,
stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 4135
Testing samples: 1034


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english")

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training data shape:", X_train_tfidf.shape)
print("Testing data shape:", X_test_tfidf.shape)

Training data shape: (4135, 7414)
Testing data shape: (1034, 7414)


In [7]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train_tfidf, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


In [8]:
y_pred_lr = lr_model.predict(X_test_tfidf)

print("Predictions completed successfully!")
print("First 10 predictions:", y_pred_lr[:10])

Predictions completed successfully!
First 10 predictions: ['ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham']


In [9]:
from sklearn.metrics import (
accuracy_score,
classification_report,
confusion_matrix,
precision_score,
recall_score,
f1_score
)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

print("Spam Precision:", precision_score(y_test, y_pred_lr, pos_label="spam"))
print("Spam Recall:", recall_score(y_test, y_pred_lr, pos_label="spam"))
print("Spam F1-score:", f1_score(y_test, y_pred_lr, pos_label="spam"))


cm_lr = confusion_matrix(y_test, y_pred_lr, labels=["ham", "spam"])

print("\nConfusion Matrix:")
print(cm_lr)

Logistic Regression Accuracy: 0.9564796905222437

Classification Report:
              precision    recall  f1-score   support

         ham       0.95      1.00      0.98       903
        spam       0.98      0.67      0.80       131

    accuracy                           0.96      1034
   macro avg       0.97      0.83      0.89      1034
weighted avg       0.96      0.96      0.95      1034

Spam Precision: 0.9777777777777777
Spam Recall: 0.6717557251908397
Spam F1-score: 0.7963800904977375

Confusion Matrix:
[[901   2]
 [ 43  88]]


In [10]:
lr_errors = pd.DataFrame({
"message": X_test,
"actual": y_test,
"predicted": y_pred_lr
})

lr_errors = lr_errors[
lr_errors["actual"] != lr_errors["predicted"]
].copy()

print("Number of incorrect predictions:", len(lr_errors))

print("\nError types:")
print(lr_errors.groupby(["actual", "predicted"]).size())

display(lr_errors.head(10))

Number of incorrect predictions: 45

Error types:
actual  predicted
ham     spam          2
spam    ham          43
dtype: int64


,message,actual,predicted
4450,Your unique user ID is 1172. For removal send ...,spam,ham
1459,Thanks for the Vote. Now sing along with the s...,spam,ham
5137,Want explicit SEX in 30 secs? Ring 02073162414...,spam,ham
521,You will recieve your tone within the next 24h...,spam,ham
621,Todays Vodafone numbers ending with 4882 are s...,spam,ham
2697,Send a logo 2 ur lover - 2 names joined by a h...,spam,ham
1876,WELL DONE! Your 4* Costa Del Sol Holiday or £5...,spam,ham
484,Congrats! 1 year special cinema pass for 2 is ...,spam,ham
3416,8007 25p 4 Alfie Moon's Children in Need song ...,spam,ham
1611,Hi if ur lookin 4 saucy daytime fun wiv busty ...,spam,ham
